# Fetal Brain Planes – Data Preparation

In [ ]:
import os
import shutil

import cv2
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from torchvision.ops import masks_to_boxes
from tqdm.notebook import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.transforms import PadToAspectRatio, Resize
from src.data.utils.utils import (
    crop,
    find_angle,
    read_image_tensor,
    save_image_tensor,
    show_pytorch_images,
)

data_dir = root / "data"
dataset_name = "FETAL_BRAIN_PLANES"
dataset_dir = data_dir / dataset_name

# Create FETAL_BRAIN_PLANES Dataset

Build the `FETAL_BRAIN_PLANES` dataset for brain-plane classification.
Starting from the cleaned `FETAL_PLANES` metadata and head-segmentation masks, this notebook produces:
- A filtered `data.csv` with consistent train/val/test splits
- A copy of all original plane images
- Rotated and cropped head images (`_crop.png`) aligned to the principal axis of the segmentation mask

## Create Dataset Directory

Create the `FETAL_BRAIN_PLANES/` output directory if it does not already exist.

In [ ]:
if not dataset_dir.exists():
    dataset_dir.mkdir(parents=True)
    print(f"Dataset directory '{dataset_name}' created successfully.")
else:
    print(f"Dataset directory '{dataset_name}' already exists.")

## Prepare CSV

Filter and clean the `FETAL_PLANES/data_fix.csv` metadata to produce the canonical label file for brain-plane classification.

### Load FETAL_PLANES CSV

Load `data_fix.csv` and keep the columns relevant to brain-plane classification: image name, patient number, train/test flag, subset assignment, brain-plane label, and the duplicate / validity / relabelling helper columns.

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data_fix.csv")
fetal_planes = fetal_planes[
    ["Image_name", "Patient_num", "Brain_plane", "New_brain_plane", "Train", "Subset", "Duplicate", "Valid"]
]
fetal_planes = fetal_planes.reset_index(drop=True)
fetal_planes

### Clean and Relabel

Apply three cleaning rules to the loaded dataframe:
1. Mark duplicate images as invalid.
2. Mark images flagged `Valid=0` as invalid.
3. Apply `New_brain_plane` corrections: rows relabelled as `'Other'` are mapped to `'Not A Brain'`.

Then drop the helper columns (`Duplicate`, `Valid`, `New_brain_plane`) so the output CSV matches the expected schema.

In [ ]:
invalid_images = 0
other_images = 0

for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    if not np.isnan(row["Duplicate"]) or row["Valid"] != 1 or isinstance(row["New_brain_plane"], str):
        fetal_planes.loc[index, "Valid"] = 0
        invalid_images += 1

    if row["Brain_plane"] == "Other":
        fetal_planes.loc[index, "Brain_plane"] = "Not A Brain"
        other_images += 1

fetal_planes = fetal_planes.drop(["New_brain_plane", "Duplicate"], axis=1)

print(f"Images: {len(fetal_planes)}")
print(f"Invalid images: {invalid_images}")
print(f"Other images: {other_images}")
fetal_planes

### Validate Subset Consistency

Cross-check that the `Subset` assignments in `FETAL_BRAIN_PLANES` match those in `FETAL_HEAD_SEGMENTATION/data.csv` for every shared image name.
Expects zero mismatches; any discrepancy indicates a drift between the two datasets.

In [ ]:
fetal_head_segmentation = pd.read_csv(f"{data_dir}/FETAL_HEAD_SEGMENTATION/data.csv")
fetal_head_segmentation

incorrect = 0
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    image_name = row["Image_name"]
    subset = row["Subset"]
    subset_2 = list(fetal_head_segmentation[fetal_head_segmentation["Image_name"] == image_name]["Subset"])
    if len(subset_2) == 1 and subset_2[0] != subset:
        incorrect += 1
        print(f"Error: {index} {row}")
        print(f"subset_2: {subset_2}")
        print()
        break

print(f"incorrect: {incorrect}")

### Statistics

Class distribution (`Brain_plane` value counts) and subset distribution (`Subset` value counts) for the cleaned dataset.

In [ ]:
fetal_planes["Subset"].value_counts()

In [ ]:
fetal_planes["Brain_plane"].value_counts()

### Save CSV

Save the cleaned dataframe as the initial `FETAL_BRAIN_PLANES/data.csv`.
The `Image_crop_name` column is not yet populated — that happens in the image generation step below.

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes

## Prepare Images

Set up the image directory and generate the rotated/cropped head images used for training the plane classifier.

### Delete Image Directory

Remove the existing `Images/` directory if it is present, to ensure the copy step below starts clean.

In [ ]:
images_dir = dataset_dir / "Images"

if images_dir.exists():
    shutil.rmtree(images_dir)
    print(f"Directory '{images_dir}' and all its contents have been removed.")
else:
    print(f"Directory '{images_dir}' does not exist.")

### Copy FETAL_PLANES Images

Copy the full `FETAL_PLANES/Images/` tree into `FETAL_BRAIN_PLANES/Images/`.
All original plane images are needed alongside the cropped head versions.

In [ ]:
images_dir = dataset_dir / "Images"

if not images_dir.exists():
    shutil.copytree(f"{data_dir}/FETAL_PLANES/Images", images_dir)
    print(f"{images_dir.stem} directory created successfully.")
else:
    print(f"{images_dir.stem} directory already exists.")

### Generate Cropped Head Images

For each brain-plane image, use the corresponding segmentation mask to rotate and crop the frame so the fetal head is centred and upright.
These `_crop.png` images are the primary input for the plane classification model.

In [ ]:
def read_mask_tensor(mask_path):
    img_path = data_dir / "FETAL_HEAD_SEGMENTATION" / mask_path
    image = read_image_tensor(img_path)
    image = image[:1, :, :]  # single-channel for mask
    image = image // 255
    return image


image_size = (192, 256)
pad_to_aspect_ration = PadToAspectRatio(image_size, fill=0)

### Load Segmentation Data

Reload `FETAL_HEAD_SEGMENTATION/data.csv` (with `Patient_num` as string) and the current `FETAL_BRAIN_PLANES/data.csv` for the crop generation pass.

In [ ]:
images_dir = dataset_dir / "Images"

segmentation = pd.read_csv(data_dir / "FETAL_HEAD_SEGMENTATION" / "data.csv", dtype={"Patient_num": str})
fetal_planes = pd.read_csv(f"{dataset_dir}/data.csv")
fetal_planes

### Crop and Rotate Brain Images

For each brain-plane row in the dataset:
1. Load the corresponding segmentation mask from `FETAL_HEAD_SEGMENTATION`.
2. Use the mask to compute the principal axis angle (`find_angle`) and bounding box.
3. Rotate the original image to align the head upright.
4. Crop to the bounding box with a padding of 5 pixels.
5. Save the result as `{image_name}_crop.png` and record the filename in `Image_crop_name`.

Only rows that have a valid mask entry in the segmentation CSV are processed.

In [ ]:
if "Image_crop_name" not in fetal_planes.columns:
    fetal_planes.insert(1, "Image_crop_name", "")
else:
    fetal_planes["Image_crop_name"] = ""


brain_planes = fetal_planes[fetal_planes["Brain_plane"] != "Not A Brain"]
for index, row in tqdm(brain_planes.iterrows(), total=len(brain_planes), desc="Process images"):
    for _, segmentation_row in segmentation[segmentation["Image_name"] == row["Image_name"]].iterrows():
        image_name = row["Image_name"]
        image_path = images_dir / f"{image_name}.png"
        image = read_image_tensor(image_path)
        image = pad_to_aspect_ration(image)

        mask_path = segmentation_row["Segmentation_path"]
        mask = read_mask_tensor(mask_path)

        angle = find_angle(mask)
        mask = pad_to_aspect_ration(mask)
        mask = TF.resize(mask, size=image.shape[1:], interpolation=T.InterpolationMode.NEAREST)
        mask = TF.rotate(mask, angle=angle, expand=True, interpolation=T.InterpolationMode.NEAREST)
        image = TF.rotate(image, angle=angle, expand=True, interpolation=T.InterpolationMode.BILINEAR)

        x1, y1, x2, y2 = masks_to_boxes(mask)[0].int()
        image = crop(image, x1, y1, x2, y2, pad=5)

        output_path = dataset_dir / "Images" / f"{image_name}_crop.png"
        save_image_tensor(image, output_path)
        fetal_planes.loc[index, "Image_crop_name"] = output_path.stem

fetal_planes

### Save Final CSV

Save `data.csv` with the `Image_crop_name` column fully populated.
This is the final label file consumed by `FetalBrainPlanesDataset` during training.

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes